# Trening Random Forest

Notebook do trenowania modelu (Google Colab). Ustaw REPO_URL w pierwszej komórce i uruchom wszystkie komórki (Runtime -> Run all).

In [ ]:
REPO_URL = ""

import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and REPO_URL:
    if not Path("Merito").exists():
        !git clone $REPO_URL Merito
    os.chdir("Merito")

ROOT = Path.cwd()
while not (ROOT / "common").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Katalog projektu:", ROOT)

In [ ]:
!pip install -q scikit-learn pandas joblib matplotlib seaborn

In [ ]:
import random
import json

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from common import config
from ml.feature_order import FEATURE_COLUMNS, LABEL_COLUMN
from ml.generate_training_data import generate_dataset

print("Klasy:", config.ALL_LABELS)
print("Kolumny cech:", FEATURE_COLUMNS)

## 1. Dane treningowe

In [ ]:
rows, counts = generate_dataset(target_per_class=300, max_iterations=200_000, seed=42)
df = pd.DataFrame(rows)
print(counts)
df.head()

In [ ]:
df[LABEL_COLUMN].value_counts().plot(kind="bar", figsize=(8, 4), title="Rozkład klas w zbiorze treningowym")
plt.ylabel("liczba przykładów")
plt.tight_layout()
plt.show()

## 2. Trening

In [ ]:
X = df[FEATURE_COLUMNS].values
y = df[LABEL_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(
    n_estimators=300, max_depth=12, random_state=42, class_weight="balanced", n_jobs=-1
)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
labels_sorted = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels_sorted)
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=labels_sorted, yticklabels=labels_sorted, cmap="Blues", ax=ax)
ax.set_xlabel("Predykcja")
ax.set_ylabel("Prawdziwa etykieta")
ax.set_title("Macierz pomyłek - Random Forest")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(clf.feature_importances_, index=FEATURE_COLUMNS).sort_values(ascending=True)
importances.plot(kind="barh", figsize=(8, 6), title="Ważność cech (Random Forest)")
plt.tight_layout()
plt.show()

## 3. Zapis modelu

Plik random_forest.joblib trzeba skopiować do ml/models/.

In [ ]:
import joblib
from pathlib import Path

MODEL_DIR = ROOT / "ml" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(clf, MODEL_DIR / "random_forest.joblib")

metadata = {
    "feature_columns": FEATURE_COLUMNS,
    "label_column": LABEL_COLUMN,
    "classes": labels_sorted,
    "test_accuracy": float((y_pred == y_test).mean()),
}
(MODEL_DIR / "model_metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
print("Zapisano model i metadane w", MODEL_DIR)

if IN_COLAB:
    from google.colab import files
    files.download(str(MODEL_DIR / "random_forest.joblib"))
    files.download(str(MODEL_DIR / "model_metadata.json"))